# 05 - Quantization & Model Variants (Intro Research Protocol)

Purpose: benchmark **latency variability** across models/variants on CPU.

This is intentionally introductory: the focus is on *measurement discipline*.


In [2]:
import os
import requests

# Resolve Ollama endpoint depending on environment
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
print('Using Ollama at:', BASE_URL)

# Diagnostic check
try:
    r = requests.get(BASE_URL, timeout=3)
    print('Server reachable. Status:', r.status_code)
    try:
        tags = requests.get(f"{BASE_URL}/api/tags", timeout=3)
        print('Tags endpoint status:', tags.status_code)
    except Exception as e:
        print('Tags endpoint check failed:', e)
except Exception as e:
    print('Server not reachable:', e)


Using Ollama at: http://host.docker.internal:11434
Server reachable. Status: 200
Tags endpoint status: 200


In [3]:
import os, json, time, statistics
import requests


In [4]:
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

def generate(prompt: str, model: str, temperature: float = 0.2) -> str:
    payload = {'model': model, 'prompt': prompt, 'temperature': temperature, 'stream': False}
    r = requests.post(f'{BASE_URL}/api/generate', json=payload, timeout=300)
    r.raise_for_status()
    return r.json().get('response', '')



In [5]:
PROMPTS = [
  'Explain attention in transformers in 6 sentences.',
  'Summarize gradient descent in 5 bullet points.',
  'Write a Python function to compute factorial.'
]
MODELS = [
  'llama3.2:3b',
  # 'gemma:2b',
  # 'phi',
]
N_RUNS = 3


In [ ]:
def bench(model: str):
    rows = []
    for prompt in PROMPTS:
        times = []
        for _ in range(N_RUNS):
            t0 = time.time()
            _ = generate(prompt, model=model)
            t1 = time.time()
            times.append(t1 - t0)
        rows.append({
            'model': model,
            'prompt': prompt,
            'n_runs': N_RUNS,
            'lat_mean_s': statistics.mean(times),
            'lat_min_s': min(times),
            'lat_max_s': max(times),
        })
    return rows

results = []
for m in MODELS:
    results.extend(bench(m))

os.makedirs('../experiments/results', exist_ok=True)
out = '../experiments/results/quantization_bench.jsonl'
with open(out, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

print('Saved', out)
results[:2]
